# Information Visualization with Altair - Complete Tutorial
## Based on UW Data Visualization Curriculum

This notebook covers all 6 essential topics for your Information Visualization project:
1. Introduction to Altair
2. Marks and Encoding
3. Data Transformation
4. Scales, Axes, and Legends
5. Multi-view Composition
6. Interaction

**Prerequisites**: Python 3.7+, pandas, altair

**How to use this notebook**: 
- Run cells sequentially using Shift+Enter
- Modify examples to experiment
- Read comments carefully for understanding

## Setup: Install Required Libraries

Run this cell first to install Altair and related libraries if you haven't already.

In [6]:
# Uncomment and run if you need to install packages
#!pip install altair pandas vega_datasets numpy

In [7]:


# Import necessary libraries
import altair as alt
import pandas as pd
import numpy as np
from vega_datasets import data  # Provides sample datasets

# Enable Altair to display charts in notebook
alt.renderers.enable('default')

# Set a maximum rows limit (Altair has a 5000 row default limit for safety)
alt.data_transformers.disable_max_rows()

print("✓ All libraries imported successfully!")
print(f"Altair version: {alt.__version__}")

✓ All libraries imported successfully!
Altair version: 5.5.0


---
# Part 1: Introduction to Altair

**Altair** is a declarative statistical visualization library for Python. It's based on Vega-Lite, which provides a grammar of graphics.

## Key Concepts:
- **Declarative**: You describe *what* you want, not *how* to draw it
- **Data-driven**: Visualizations are built from data
- **Composable**: Complex visualizations from simple building blocks

In [8]:
# Load a sample dataset: cars data
cars = data.cars()

# Let's explore the data first
print("First 5 rows of the cars dataset:")
print(cars.head())
print(f"\nDataset shape: {cars.shape} (rows, columns)")
print(f"\nColumn names: {list(cars.columns)}")

First 5 rows of the cars dataset:
                        Name  Miles_per_Gallon  Cylinders  Displacement  \
0  chevrolet chevelle malibu              18.0          8         307.0   
1          buick skylark 320              15.0          8         350.0   
2         plymouth satellite              18.0          8         318.0   
3              amc rebel sst              16.0          8         304.0   
4                ford torino              17.0          8         302.0   

   Horsepower  Weight_in_lbs  Acceleration       Year Origin  
0       130.0           3504          12.0 1970-01-01    USA  
1       165.0           3693          11.5 1970-01-01    USA  
2       150.0           3436          11.0 1970-01-01    USA  
3       150.0           3433          12.0 1970-01-01    USA  
4       140.0           3449          10.5 1970-01-01    USA  

Dataset shape: (406, 9) (rows, columns)

Column names: ['Name', 'Miles_per_Gallon', 'Cylinders', 'Displacement', 'Horsepower', 'Weight_i

In [9]:
# Create your first Altair chart: A simple scatter plot
# The basic pattern: Chart(data).mark_type().encode(channels)

chart1 = alt.Chart(cars).mark_point().encode(
    x='Horsepower',  # Horizontal position
    y='Miles_per_Gallon'  # Vertical position
)

# Display the chart
chart1

alt.Chart(...)

In [10]:
# Enhance the chart with more encoding channels
# We can encode additional data dimensions using color, size, shape, etc.

chart2 = alt.Chart(cars).mark_point().encode(
    x='Horsepower',
    y='Miles_per_Gallon',
    color='Origin',  # Color points by country of origin
    size='Weight_in_lbs',  # Size points by weight
    tooltip=['Name', 'Horsepower', 'Miles_per_Gallon', 'Origin']  # Show info on hover
).properties(
    width=600,
    height=400,
    title='Car Performance: Horsepower vs MPG'
)

chart2

alt.Chart(...)

---
# Part 2: Marks and Encoding

## Marks
**Marks** are the visual elements that represent data. Common marks include:
- `mark_point()` - scatter plot points
- `mark_bar()` - bar charts
- `mark_line()` - line charts
- `mark_area()` - filled areas
- `mark_rect()` - rectangles (for heatmaps)
- `mark_text()` - text labels
- `mark_tick()` - tick marks

## Encoding Channels
**Encoding channels** map data to visual properties:
- Position: `x`, `y`
- Color: `color`
- Size: `size`
- Shape: `shape`
- Opacity: `opacity`
- Text: `text`, `tooltip`

In [11]:
# Example 1: Bar chart
# Count the number of cars by origin

bar_chart = alt.Chart(cars).mark_bar().encode(
    x='Origin',  # Categorical variable on x-axis
    y='count()',  # Count the number of records
    color='Origin'  # Color bars by origin
).properties(
    title='Number of Cars by Origin'
)

bar_chart

alt.Chart(...)

In [12]:
# Example 2: Line chart
# Average MPG over time

line_chart = alt.Chart(cars).mark_line(point=True).encode(
    x='Year:O',  # ':O' specifies ordinal (categorical) data type
    y='average(Miles_per_Gallon)',  # Aggregate function
    tooltip=['Year', 'average(Miles_per_Gallon)']
).properties(
    title='Average MPG Over Time',
    width=600
)

line_chart

alt.Chart(...)

In [13]:
# Example 3: Understanding data types in Altair
# Data types affect how Altair interprets and displays data
# - Q: Quantitative (continuous numbers)
# - O: Ordinal (ordered categories)
# - N: Nominal (unordered categories)
# - T: Temporal (dates/times)

# Compare different encodings of the same data

# Cylinder as quantitative (continuous scale)
chart_q = alt.Chart(cars).mark_point().encode(
    x='Cylinders:Q',  # Quantitative - continuous scale
    y='Miles_per_Gallon',
    color='Cylinders:Q'
).properties(
    title='Cylinders as Quantitative',
    width=250
)

# Cylinder as ordinal (discrete ordered scale)
chart_o = alt.Chart(cars).mark_point().encode(
    x='Cylinders:O',  # Ordinal - discrete categories
    y='Miles_per_Gallon',
    color='Cylinders:O'
).properties(
    title='Cylinders as Ordinal',
    width=250
)

# Display side by side (we'll learn more about composition later)
chart_q | chart_o

alt.HConcatChart(...)

---
# Part 3: Data Transformation

Altair can transform data before visualization without modifying the original dataframe.

## Common Transformations:
- **Aggregation**: `count()`, `mean()`, `sum()`, `median()`, etc.
- **Filtering**: `transform_filter()`
- **Calculation**: `transform_calculate()`
- **Binning**: Grouping continuous data into bins
- **Window**: Running calculations (moving averages, etc.)

In [14]:
# Example 1: Filtering data
# Show only cars with good fuel efficiency (MPG > 30)

filtered_chart = alt.Chart(cars).mark_point().encode(
    x='Horsepower',
    y='Miles_per_Gallon',
    color='Origin',
    tooltip=['Name', 'Miles_per_Gallon']
).transform_filter(
    alt.datum.Miles_per_Gallon > 30  # Filter condition
).properties(
    title='Fuel-Efficient Cars (MPG > 30)'
)

filtered_chart

alt.Chart(...)

In [15]:
# Example 2: Calculate new fields
# Create a new field: Horsepower per cylinder

calculated_chart = alt.Chart(cars).mark_point().encode(
    x='Cylinders:O',
    y='hp_per_cylinder:Q',
    color='Origin',
    tooltip=['Name', 'Horsepower', 'Cylinders', 'hp_per_cylinder:Q']
).transform_calculate(
    # Create new calculated field
    hp_per_cylinder='datum.Horsepower / datum.Cylinders'
).properties(
    title='Horsepower per Cylinder by Number of Cylinders'
)

calculated_chart

alt.Chart(...)

In [16]:
# Example 3: Binning continuous data
# Create histogram of horsepower

histogram = alt.Chart(cars).mark_bar().encode(
    x=alt.X('Horsepower:Q', bin=alt.Bin(maxbins=20)),  # Bin the data
    y='count()',
    tooltip=['count()']
).properties(
    title='Distribution of Horsepower',
    width=600
)

histogram

alt.Chart(...)

In [17]:
# Example 4: Aggregation
# Average MPG by number of cylinders and origin

grouped_bar = alt.Chart(cars).mark_bar().encode(
    x='Cylinders:O',
    y='average(Miles_per_Gallon)',
    color='Origin',
    column='Origin'  # Create separate plots for each origin
).properties(
    title='Average MPG by Cylinders and Origin',
    width=150
)

grouped_bar

alt.Chart(...)

---
# Part 4: Scales, Axes, and Legends

**Scales** map data values to visual values (positions, colors, sizes).

**Axes** provide reference marks and labels for position scales.

**Legends** provide reference marks and labels for color, size, shape scales.

## Customization Options:
- Scale types: linear, log, sqrt, time, etc.
- Scale domains and ranges
- Axis titles, labels, and formatting
- Legend positioning and styling

In [18]:
# Example 1: Customizing scales
# Use logarithmic scale for better visualization

scale_chart = alt.Chart(cars).mark_point().encode(
    x=alt.X('Horsepower', 
            scale=alt.Scale(type='log'),  # Logarithmic scale
            axis=alt.Axis(title='Horsepower (log scale)')
    ),
    y=alt.Y('Miles_per_Gallon',
            scale=alt.Scale(zero=False),  # Don't force zero baseline
            axis=alt.Axis(title='Miles per Gallon')
    ),
    color=alt.Color('Origin',
                   legend=alt.Legend(title='Country of Origin')
    ),
    tooltip=['Name', 'Horsepower', 'Miles_per_Gallon']
).properties(
    title='Horsepower vs MPG (Log Scale)',
    width=600,
    height=400
)

scale_chart

alt.Chart(...)

In [19]:
# Example 2: Custom color scales
# Define specific colors for each origin

custom_color_chart = alt.Chart(cars).mark_point(filled=True, size=100).encode(
    x='Horsepower',
    y='Miles_per_Gallon',
    color=alt.Color('Origin',
                   scale=alt.Scale(
                       domain=['USA', 'Europe', 'Japan'],  # Define categories
                       range=['#e74c3c', '#3498db', '#2ecc71']  # Define colors
                   ),
                   legend=alt.Legend(
                       title='Origin',
                       orient='right',  # Position legend on right
                       titleFontSize=14,
                       labelFontSize=12
                   )
    ),
    tooltip=['Name', 'Origin', 'Horsepower', 'Miles_per_Gallon']
).properties(
    title='Custom Color Scale Example',
    width=600
)

custom_color_chart

alt.Chart(...)

In [20]:
# Example 3: Customizing axes
# Format axis labels and grid

axis_chart = alt.Chart(cars).mark_point().encode(
    x=alt.X('Horsepower',
           axis=alt.Axis(
               title='Horsepower (HP)',
               titleFontSize=14,
               titleFontWeight='bold',
               labelAngle=-45,  # Rotate labels
               grid=True,  # Show grid lines
               gridColor='lightgray'
           )
    ),
    y=alt.Y('Miles_per_Gallon',
           axis=alt.Axis(
               title='Fuel Efficiency (MPG)',
               titleFontSize=14,
               titleFontWeight='bold',
               format='.1f',  # Format numbers to 1 decimal place
               grid=True
           )
    ),
    color='Origin',
    tooltip=['Name', 'Horsepower', 'Miles_per_Gallon']
).properties(
    title='Customized Axes Example',
    width=600,
    height=400
)

axis_chart

alt.Chart(...)

In [21]:
# Example 4: Domain and range customization
# Control the exact mapping of data to visual properties

domain_chart = alt.Chart(cars).mark_point().encode(
    x=alt.X('Horsepower',
           scale=alt.Scale(domain=[0, 250])  # Set x-axis range
    ),
    y=alt.Y('Miles_per_Gallon',
           scale=alt.Scale(domain=[0, 50])  # Set y-axis range
    ),
    size=alt.Size('Weight_in_lbs',
                 scale=alt.Scale(range=[10, 500]),  # Control size range
                 legend=alt.Legend(title='Weight (lbs)')
    ),
    color='Origin',
    tooltip=['Name', 'Horsepower', 'Miles_per_Gallon', 'Weight_in_lbs']
).properties(
    title='Custom Domain and Range',
    width=600
)

domain_chart

alt.Chart(...)

---
# Part 5: Multi-View Composition

Combine multiple charts to create complex visualizations.

## Composition Operators:
- `chart1 | chart2` - Horizontal concatenation (side by side)
- `chart1 & chart2` - Vertical concatenation (stacked)
- `alt.layer(chart1, chart2)` - Layer charts on top of each other
- `alt.hconcat(chart1, chart2)` - Horizontal concatenation (alternative)
- `alt.vconcat(chart1, chart2)` - Vertical concatenation (alternative)
- `alt.repeat()` - Create multiple views with different fields
- `alt.facet()` - Create small multiples based on data

In [22]:
# Example 1: Horizontal and vertical concatenation
# Create a dashboard with multiple views

# Scatter plot
scatter = alt.Chart(cars).mark_point().encode(
    x='Horsepower',
    y='Miles_per_Gallon',
    color='Origin',
    tooltip=['Name', 'Horsepower', 'Miles_per_Gallon']
).properties(
    width=300,
    height=250,
    title='Scatter: HP vs MPG'
)

# Histogram of horsepower
hist_hp = alt.Chart(cars).mark_bar().encode(
    x=alt.X('Horsepower', bin=alt.Bin(maxbins=15)),
    y='count()',
    color='Origin'
).properties(
    width=300,
    height=250,
    title='Distribution of HP'
)

# Bar chart of average MPG by origin
bar_mpg = alt.Chart(cars).mark_bar().encode(
    x='Origin',
    y='average(Miles_per_Gallon)',
    color='Origin',
    tooltip=['Origin', 'average(Miles_per_Gallon)']
).properties(
    width=300,
    height=250,
    title='Avg MPG by Origin'
)

# Combine: scatter and histogram side by side, bar chart below
dashboard = (scatter | hist_hp) & bar_mpg

dashboard

alt.VConcatChart(...)

In [23]:
# Example 2: Layering charts
# Combine a scatter plot with a trend line

# Base scatter plot
points = alt.Chart(cars).mark_point().encode(
    x='Horsepower',
    y='Miles_per_Gallon',
    color='Origin',
    tooltip=['Name', 'Horsepower', 'Miles_per_Gallon']
)

# Add a regression line
line = alt.Chart(cars).mark_line(color='red', size=3).encode(
    x='Horsepower',
    y='Miles_per_Gallon'
).transform_regression(
    'Horsepower', 'Miles_per_Gallon'  # Calculate regression
)

# Layer them together
layered_chart = (points + line).properties(
    title='Scatter Plot with Regression Line',
    width=600,
    height=400
)

layered_chart

alt.LayerChart(...)

In [24]:
# Example 3: Repeat pattern
# Create a scatter plot matrix (SPLOM) to compare multiple variables

splom = alt.Chart(cars).mark_point(size=20, opacity=0.5).encode(
    x=alt.X(alt.repeat('column'), type='quantitative'),
    y=alt.Y(alt.repeat('row'), type='quantitative'),
    color='Origin:N',
    tooltip=['Name', 'Origin']
).properties(
    width=200,
    height=200
).repeat(
    row=['Horsepower', 'Acceleration'],
    column=['Miles_per_Gallon', 'Weight_in_lbs']
)

splom

alt.RepeatChart(...)

In [25]:
# Example 4: Faceting (Small Multiples)
# Create separate charts for each category

faceted_chart = alt.Chart(cars).mark_point().encode(
    x='Horsepower',
    y='Miles_per_Gallon',
    color='Origin',
    tooltip=['Name', 'Horsepower', 'Miles_per_Gallon']
).properties(
    width=250,
    height=250
).facet(
    column='Origin',  # Create separate chart for each origin
    title='HP vs MPG by Origin (Faceted)'
)

faceted_chart

alt.FacetChart(...)

---
# Part 6: Interaction

Interactive visualizations allow users to explore data dynamically.

## Interaction Techniques:
- **Selection**: Click or drag to select data points
- **Binding**: Connect selections to UI elements (dropdowns, sliders)
- **Conditional Encoding**: Change appearance based on selection
- **Zooming and Panning**: Explore different regions
- **Tooltips**: Show details on hover (already seen in previous examples)
- **Linked Views**: Connect multiple charts

In [26]:
# Example 1: Basic selection - Click to highlight
# Create an interval selection (drag to select)

# Define a selection
selection = alt.selection_point(fields=['Origin'])

# Create chart with conditional encoding based on selection
interactive_chart = alt.Chart(cars).mark_point(size=100).encode(
    x='Horsepower',
    y='Miles_per_Gallon',
    color=alt.condition(selection, 'Origin', alt.value('lightgray')),
    opacity=alt.condition(selection, alt.value(1.0), alt.value(0.2)),
    tooltip=['Name', 'Origin', 'Horsepower', 'Miles_per_Gallon']
).add_params(
    selection  # Add selection to chart
).properties(
    title='Click on Legend to Filter Data',
    width=600,
    height=400
).interactive()  # Enable zooming and panning

interactive_chart

alt.Chart(...)

In [27]:
# Example 2: Interval selection (brush)
# Drag to select a region

# Define interval selection
brush = alt.selection_interval()

brush_chart = alt.Chart(cars).mark_point().encode(
    x='Horsepower',
    y='Miles_per_Gallon',
    color=alt.condition(brush, 'Origin', alt.value('lightgray')),
    size=alt.condition(brush, alt.value(100), alt.value(30)),
    tooltip=['Name', 'Origin', 'Horsepower', 'Miles_per_Gallon']
).add_params(
    brush
).properties(
    title='Drag to Select Region (Brush Selection)',
    width=600,
    height=400
)

brush_chart

alt.Chart(...)

In [28]:
# Example 3: Linked views with shared selection
# Selection in one chart affects another

# Define a shared selection
brush_linked = alt.selection_interval()

# First chart: Horsepower vs MPG
chart1_linked = alt.Chart(cars).mark_point().encode(
    x='Horsepower',
    y='Miles_per_Gallon',
    color=alt.condition(brush_linked, 'Origin', alt.value('lightgray')),
    tooltip=['Name', 'Horsepower', 'Miles_per_Gallon']
).add_params(
    brush_linked
).properties(
    width=300,
    height=300,
    title='Select Region Here'
)

# Second chart: Weight vs Acceleration (responds to selection in first chart)
chart2_linked = alt.Chart(cars).mark_point().encode(
    x='Weight_in_lbs',
    y='Acceleration',
    color=alt.condition(brush_linked, 'Origin', alt.value('lightgray')),
    tooltip=['Name', 'Weight_in_lbs', 'Acceleration']
).properties(
    width=300,
    height=300,
    title='Linked View'
)

# Combine side by side
linked_views = chart1_linked | chart2_linked

linked_views

alt.HConcatChart(...)

In [29]:
# Example 4: Dropdown binding
# Use a dropdown menu to filter data

# Create dropdown selection
origin_dropdown = alt.binding_select(
    options=[None, 'USA', 'Europe', 'Japan'],  # None shows all
    name='Origin: '  # Label for dropdown
)

origin_selection = alt.selection_point(
    fields=['Origin'],
    bind=origin_dropdown
)

dropdown_chart = alt.Chart(cars).mark_point(size=100).encode(
    x='Horsepower',
    y='Miles_per_Gallon',
    color='Origin',
    opacity=alt.condition(origin_selection, alt.value(1.0), alt.value(0.1)),
    tooltip=['Name', 'Origin', 'Horsepower', 'Miles_per_Gallon']
).add_params(
    origin_selection
).properties(
    title='Filter by Origin Using Dropdown',
    width=600,
    height=400
)

dropdown_chart

alt.Chart(...)

In [30]:
# Example 5: Slider binding
# Use a slider to filter by year

# Get min and max years
min_year = int(cars['Year'].min())
max_year = int(cars['Year'].max())

# Create slider
year_slider = alt.binding_range(
    min=min_year,
    max=max_year,
    step=1,
    name='Year: '
)

year_selection = alt.param(
    name='year_param',
    value=min_year,
    bind=year_slider
)

slider_chart = alt.Chart(cars).mark_point(size=100).encode(
    x='Horsepower',
    y='Miles_per_Gallon',
    color='Origin',
    tooltip=['Name', 'Year', 'Origin', 'Horsepower', 'Miles_per_Gallon']
).add_params(
    year_selection
).transform_filter(
    alt.datum.Year >= year_selection  # Filter based on slider value
).properties(
    title='Filter by Year Using Slider',
    width=600,
    height=400
)

slider_chart

TypeError: int() argument must be a string, a bytes-like object or a real number, not 'Timestamp'

In [ ]:
# Example 6: Advanced - Dashboard with multiple interactions
# Combining everything we've learned

# Shared selection
brush_adv = alt.selection_interval()

# Main scatter plot with brush selection
main_scatter = alt.Chart(cars).mark_point().encode(
    x='Horsepower',
    y='Miles_per_Gallon',
    color=alt.condition(brush_adv, 'Origin', alt.value('lightgray')),
    tooltip=['Name', 'Origin', 'Horsepower', 'Miles_per_Gallon']
).add_params(
    brush_adv
).properties(
    width=400,
    height=300,
    title='Main View: Select Points'
)

# Histogram showing distribution of selected points
hist_selected = alt.Chart(cars).mark_bar().encode(
    x=alt.X('Miles_per_Gallon', bin=alt.Bin(maxbins=20)),
    y='count()',
    color='Origin'
).transform_filter(
    brush_adv  # Only show selected data
).properties(
    width=400,
    height=150,
    title='MPG Distribution of Selected'
)

# Bar chart showing average by origin for selected
bar_selected = alt.Chart(cars).mark_bar().encode(
    x='Origin',
    y='average(Miles_per_Gallon)',
    color='Origin',
    tooltip=['Origin', 'average(Miles_per_Gallon)', 'count()']
).transform_filter(
    brush_adv
).properties(
    width=400,
    height=150,
    title='Average MPG by Origin (Selected)'
)

# Combine into dashboard
advanced_dashboard = main_scatter & (hist_selected | bar_selected)

advanced_dashboard

---
# Summary and Next Steps

Congratulations! You've completed the Altair tutorial covering all 6 essential topics:

✓ **Introduction**: Basic chart structure and syntax
✓ **Marks & Encoding**: Visual elements and data mapping
✓ **Data Transformation**: Filtering, calculating, aggregating
✓ **Scales, Axes & Legends**: Customizing visual mappings
✓ **Multi-view Composition**: Combining multiple charts
✓ **Interaction**: Creating dynamic, explorable visualizations

## Practice Exercises:

1. **Load your own dataset** and create visualizations
2. **Combine techniques**: Create a dashboard with filtering, linked views, and custom styling
3. **Explore other datasets** from `vega_datasets` (movies, weather, stocks, etc.)
4. **Read the documentation**: https://altair-viz.github.io/

## Tips for Your Project:

- Start simple, then add complexity
- Think about what story you want to tell with your data
- Use interaction to let users explore
- Test your visualizations with others
- Document your design choices in your report

## Common Issues:

- **Charts not displaying**: Make sure renderers are enabled
- **Too many data points**: Use `alt.data_transformers.disable_max_rows()`
- **Slow performance**: Consider aggregating data or sampling
- **Encoding errors**: Check data types (Q, N, O, T)

Good luck with your Information Visualization project!

In [ ]:
# Bonus: Save charts to HTML
# You can save any chart as an interactive HTML file

# Example: Save the advanced dashboard
# advanced_dashboard.save('my_dashboard.html')

# Or save as JSON spec
# advanced_dashboard.save('my_chart_spec.json')

# Or save as PNG (requires additional dependencies: altair_saver, selenium)
# advanced_dashboard.save('my_chart.png')

print("To save charts, uncomment the code above and run this cell.")